# Model Analysis Project -- section 4.1

The revenue function `tax_revenue()` (eq. 5). This notebook is a stand-alone
excerpt: it runs on its own as long as `Consumer.py` and `Government.py` are in
the same folder.

In [24]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize

from consumer1 import ConsumerClass
from Government import GovernmentClass

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 4. Taxes

The government has two kinds of instrument, both placed on top of the consumer
from sections 1-3:

1. a **lump-sum tax** `T`, taken straight out of income: `I = I_pre - T`
2. **product taxes** `tau1, tau2, tau3`, which raise the prices the consumer
   pays: `p_j = (1+tau_j) * p_j_pre`

`GovernmentClass` inherits from `ConsumerClass`, so `set_taxes()` simply writes
new values into `par.p1, par.p2, par.p3, par.I`. After that, `solve()`,
`shares()` and `quantities()` automatically refer to the situation *with* taxes,
without changing a single line of the consumer code. `sync_pre_tax()` stores the
pre-tax situation (`p1_pre, p2_pre, p3_pre, I_pre`) in `__init__`, and it does so
*after* the parameters have been updated, so a changed price is picked up
correctly.

### 4.1 Revenue: `tax_revenue()`

Total tax revenue is (eq. 5)

$$ R = T + \sum_{j=1}^{3} \tau_j \, p_j^{\text{pre}} \, x_j^{\star} $$

where $x_j^{\star}$ is what the consumer buys *given* the taxes. The
implementation in `Government.py` is:

```python
def tax_revenue(self,opt=None):

    par = self.par

    # a. the quantities the consumer buys under the current taxes
    if opt is None: opt = self.solve(do_print=False)
    x1,x2,x3 = self.quantities(opt.s1,opt.w)

    # b. lump-sum tax + product tax on each good
    R = par.T
    R += par.tau1*par.p1_pre*x1
    R += par.tau2*par.p2_pre*x2
    R += par.tau3*par.p3_pre*x3

    return R
```

Two details are worth noting:

**The seller price, not the consumer price.** The rate multiplies `p_j_pre` and
not `p_j = (1+tau_j) p_j_pre`. The consumer pays `p_j*x_j` for good j; of that,
`p_j_pre*x_j` goes to the seller and `tau_j*p_j_pre*x_j` goes to the government.
Using the consumer price would overstate revenue by the factor `(1+tau_j)` --
that is 50 percent too much at a tax rate of 50 percent.

**`opt` is passed in.** The function solves the consumer's problem itself if it
is not given a solution. When a solution is already at hand from `solve()`,
passing it in saves an entire optimization -- which matters in 4.2-4.5, where
`tax_revenue()` is called thousands of times.

Below, the implementation is checked in four ways: against eq. (5) computed by
hand, against zero revenue with no taxes, against `R = T` for a lump-sum tax, and
against the analytical expression `R = tau*I/(1+tau)` for a uniform tax on all
three goods.

In [25]:
# a. the two calibrations, same pattern as in section 1
gov = {}
gov['complements'] = GovernmentClass()                       # sigma_B = 0.40
gov['substitutes'] = GovernmentClass(par={'sigma_B':3.0})    # sigma_B = 3.00

# b. check that sync_pre_tax() stored the pre-tax situation, and that no tax is set yet
for name, model in gov.items():
    par = model.par
    print(f'{name}: sigma_B={par.sigma_B:.2f}, '
          f'p1_pre={par.p1_pre:.2f}, p2_pre={par.p2_pre:.2f}, p3_pre={par.p3_pre:.2f}, '
          f'I_pre={par.I_pre:.2f}, T={par.T:.2f}, '
          f'tau=({par.tau1:.2f},{par.tau2:.2f},{par.tau3:.2f})')

NameError: name 'SimpleNamespace' is not defined

The five tax experiments are collected in a list of dictionaries, so that one
loop can run all of them for both calibrations.

In [ ]:
# a. the taxes to try out: a name + the arguments for set_taxes()
experiments = []
experiments.append({'name':'no taxes',       'T':0.00,'tau1':0.00,'tau2':0.00,'tau3':0.00})
experiments.append({'name':'lump-sum T=1',   'T':1.00,'tau1':0.00,'tau2':0.00,'tau3':0.00})
experiments.append({'name':'food tau1=0.5',  'T':0.00,'tau1':0.50,'tau2':0.00,'tau3':0.00})
experiments.append({'name':'train tau3=0.5', 'T':0.00,'tau1':0.00,'tau2':0.00,'tau3':0.50})
experiments.append({'name':'uniform tau=0.25','T':0.00,'tau1':0.25,'tau2':0.25,'tau3':0.25})

In [ ]:
tables = {}
for name, model in gov.items():

    par = model.par
    rows = []

    for exp in experiments:

        # a. set the taxes and solve the consumer's problem under them
        model.set_taxes(T=exp['T'],tau1=exp['tau1'],tau2=exp['tau2'],tau3=exp['tau3'])
        opt = model.solve(do_print=False)
        x1,x2,x3 = model.quantities(opt.s1,opt.w)

        # b. revenue from the method
        R = model.tax_revenue(opt)

        # c. the same computed by hand from eq. (5), with the seller prices
        R_hand = par.T
        R_hand += par.tau1*par.p1_pre*x1
        R_hand += par.tau2*par.p2_pre*x2
        R_hand += par.tau3*par.p3_pre*x3

        # d. what one would wrongly get with the prices the consumer pays
        R_wrong = par.T
        R_wrong += par.tau1*par.p1*x1
        R_wrong += par.tau2*par.p2*x2
        R_wrong += par.tau3*par.p3*x3

        # e. check 1: the method must hit eq. (5) exactly
        assert np.isclose(R,R_hand), 'tax_revenue() and eq. (5) must agree'

        rows.append({'experiment':exp['name'],'x1':x1,'x2':x2,'x3':x3,
                     'R':R,'R by hand':R_hand,'R at consumer prices':R_wrong,'u':opt.u})

    # f. clean up after the calibration, and store the table
    model.set_taxes()
    tables[name] = pd.DataFrame(rows)

AttributeError: 'types.SimpleNamespace' object has no attribute 's1'

In [ ]:
for name, table in tables.items():
    print(name)
    display(table.round(4))

### Three analytical checks

Three cases have an answer that is known in advance and does not depend on the
optimizer:

- **no taxes:** `R = 0`
- **lump-sum:** `R = T` -- the consumer can do nothing to avoid it
- **a uniform tax on all three goods:** all income still goes to consumption, so
  `sum_j p_j*x_j = I` and therefore `sum_j p_j_pre*x_j = I/(1+tau)`. Revenue must
  then be `R = tau*I/(1+tau)` -- completely independent of `sigma_A`, `sigma_B`,
  `alpha`, `beta` and the prices.

The last one is the sharpest test that seller prices are being used: with
consumer prices one would get `tau*I` instead.

In [ ]:
for name, model in gov.items():

    par = model.par

    # a. no taxes -> no revenue
    R0,u0 = model.revenue_and_utility(0.0,goods=(2,))
    assert np.isclose(R0,0.0), 'revenue must be zero without taxes'

    # b. lump-sum -> revenue is exactly T
    T = 1.0
    R_T,u_T = model.revenue_and_utility_lump_sum(T)
    assert np.isclose(R_T,T), 'lump-sum revenue must be exactly T'

    # c. uniform tax on all three goods -> tau*I/(1+tau)
    tau = 0.25
    R_uni,u_uni = model.revenue_and_utility(tau,goods=(1,2,3))
    R_analytic = tau*par.I/(1+tau)
    assert np.isclose(R_uni,R_analytic), 'a uniform tax must give tau*I/(1+tau)'

    print(f'{name}: R(0)={R0:.6f} | lump-sum R={R_T:.6f} (T={T:.2f}) | '
          f'uniform R={R_uni:.6f} vs tau*I/(1+tau)={R_analytic:.6f}')

    # d. clean up, so the next calibration and the next question start tax-free
    model.set_taxes()

In [ ]:
# a. final control: both models are back at the starting point before 4.2
for name, model in gov.items():
    par = model.par
    print(f'{name}: p1={par.p1:.2f}, p2={par.p2:.2f}, p3={par.p3:.2f}, I={par.I:.2f}, '
          f'T={par.T:.2f}, tau=({par.tau1:.2f},{par.tau2:.2f},{par.tau3:.2f})')

### Interpretation

**The implementation is correct.** `tax_revenue()` matches eq. (5) computed by
hand in all ten cases (five experiments times two calibrations), and the three
analytical checks pass: `R = 0` without taxes, `R = 1.0000` for a lump-sum tax of
`T = 1`, and `R = 2.0000 = 0.25*10/1.25` for the uniform tax in *both*
calibrations.

**The seller price is the right price.** The column `R at consumer prices` shows
what one would have got by applying the rate to the price the consumer pays. For
the train tax, `p3_pre = 1.50` and `p3 = 2.25`, and the error is exactly a factor
1.5: 0.9832 against 1.4748 in the complements calibration. The consumer's outlay
on train tickets is `2.25*x3`, of which `1.50*x3` is the seller's turnover and
only `0.75*x3` is the government's. With the consumer price the tax would be
counted twice, and all of section 4 would systematically overstate what the
government can raise -- worst at high rates, which is exactly where it decides
whether the revenue curve has a top.

**A lump-sum tax does not distort.** At `T = 1` all three quantities fall by
exactly 10 percent (5.3562 -> 4.8206, 2.0408 -> 1.8368, 1.7353 -> 1.5618 in the
complements calibration): the budget shares are unchanged, because only income
has shrunk. No relative price has moved, so there is no substitution -- only an
income effect. That is why the lump-sum tax is the benchmark in 4.4.

**The calibration decides how much a narrow tax can raise.** The same 50 percent
tax on train tickets raises 0.9832 in the complements calibration, but only
0.2565 among substitutes -- less than a third. The explanation is in the `x3`
column: with `sigma_B = 0.40` bus and train are hard to swap, so consumption only
falls from 1.7353 to 1.3110 and the consumer pays the tax. With `sigma_B = 3.00`
he escapes into the bus (`x2` rises from 3.1969 to 3.8950) and train consumption
collapses from 0.9472 to 0.3419. A narrow tax base that is easy to substitute
away from is a poor tax base -- and that is precisely the mechanism behind the
peaks in 4.3.

**The breadth of the base matters more than the rate.** The tax on food raises
1.85 at a rate of 50 percent, while the uniform tax raises 2.00 at a rate of only
25 percent. Food is more than half the budget and has just `sigma_A = 0.80` to
escape with, whereas the uniform tax cannot be avoided by substitution at all --
only by consuming less. Utility, on the other hand, ends up almost the same
(2.7265 against 2.7213 for complements), so the cheap revenue is not free; that
comparison is taken up systematically in 4.4.